# Synthetic Data Generation Script

In [12]:
import json
import random
import re
from datetime import datetime, timedelta
from collections import defaultdict

# ==========================================================
# STRICT TOOL ROUTER SYNTHETIC DATASET GENERATOR
# Generates 700 HIGHLY DIVERSE + BALANCED samples
# ----------------------------------------------------------
# Tools:
# 1. weather(location, unit)
# 2. calendar(action, date, title)
# 3. convert(value, from_unit, to_unit)
# 4. currency(amount, from_ISO3, to_ISO3)
# 5. sql(query)
#
# Final Mix:
# - 350 VALID TOOL CALLS
# - 350 REFUSALS / OUT OF SCOPE
# Balanced across all 5 tools
# ==========================================================

random.seed(42)

# ==========================================================
# CONFIG
# ==========================================================
TOTAL_SAMPLES = 2000
VALID_SAMPLES = 1500
REFUSAL_SAMPLES = 500
PER_TOOL = VALID_SAMPLES // 5   # 70 each tool

OUTPUT_FILE = "tool_router_700_balanced.jsonl"

# ==========================================================
# DATA POOLS
# ==========================================================
cities = [
    "London","New York","Tokyo","Paris","Berlin","Dubai","Sydney","Toronto",
    "Singapore","Lahore","Karachi","Islamabad","Mumbai","Delhi","Istanbul",
    "Bangkok","Madrid","Rome","Chicago","Seoul","Beijing","Cairo","Doha"
]

temp_units = ["C", "F"]

calendar_titles = [
    "Team Meeting","Doctor Appointment","Project Review","Lunch with Client",
    "Gym Session","Interview","Flight Reminder","Birthday Dinner",
    "Code Review","Marketing Call","Sales Sync","Dentist Visit"
]

measure_units = [
    "km","miles","meters","feet","kg","lbs","grams","ounces",
    "liters","gallons","cm","inches"
]

currencies = [
    "USD","EUR","GBP","JPY","AUD","CAD","PKR","INR","CNY","CHF",
    "AED","SAR","TRY","KRW","SGD"
]

sql_tables = [
    "users","orders","employees","products","inventory",
    "payments","customers","tickets","sales","shipments"
]

# ==========================================================
# REFUSAL REQUESTS
# ==========================================================
refusal_prompts = [
    "Tell me a joke.",
    "Write python code for a snake game.",
    "Turn off my bedroom lights.",
    "Open Spotify.",
    "Who will win the world cup?",
    "Translate this paragraph to German.",
    "Book a flight to Dubai.",
    "Hack my neighbor wifi.",
    "Give me relationship advice.",
    "Generate a logo for my startup.",
    "Summarize this PDF.",
    "Explain quantum mechanics simply.",
    "Call mom now.",
    "Post this tweet for me.",
    "What's your favorite movie?",
    "Make me rich fast.",
    "Read my mind.",
    "Set alarm for tomorrow.",
    "Play baby shark.",
    "Recommend Netflix series.",
    "Create website homepage.",
    "Tell me horoscope.",
    "Sing me a song.",
    "Open camera app.",
    "Design tattoo idea."
]

refusal_responses = [
    "I can only help with weather, calendar, unit conversion, currency conversion, or SQL queries.",
    "That request is outside my available tools. I only support weather, calendar, convert, currency, and sql.",
    "I’m limited to these tools: weather, calendar, convert, currency, and sql.",
    "I can't complete that request because it is not supported by my available tools.",
]

# ==========================================================
# UTILS
# ==========================================================
def rand_date():
    d = datetime.now() + timedelta(days=random.randint(1, 120))
    return d.strftime("%Y-%m-%d")

def escape_json(s):
    return s.replace("\\", "\\\\").replace('"', '\\"')

def write_jsonl_line(user_text, assistant_text):
    return {
        "messages": [
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": assistant_text}
        ]
    }

# ==========================================================
# TOOL BUILDERS
# ==========================================================
def weather_sample():
    city = random.choice(cities)
    unit = random.choice(temp_units)

    prompts = [
        f"Weather in {city}",
        f"What's the temp in {city}?",
        f"How hot is it in {city}?",
        f"{city} weather pls",
        f"Forecast for {city}",
        f"Mausam {city} ka?",
        f"clima en {city}",
        f"{city} temp in {unit}",
        f"is it raining in {city}"
    ]

    user = random.choice(prompts)

    tool = (
        f'<tool_call>{{"tool":"weather","args":{{'
        f'"location":"{city}","unit":"{unit}"'
        f'}}}}</tool_call>'
    )
    return user, tool


def calendar_sample():
    action = random.choice(["create", "list"])
    date = rand_date()

    if action == "create":
        title = random.choice(calendar_titles)

        prompts = [
            f"Add {title} on {date}",
            f"Create event {title} {date}",
            f"Schedule {title} for {date}",
            f"Put {title} in my calendar {date}",
            f"Agendar {title} {date}",
            f"{date} pe {title} laga do"
        ]

        tool = (
            f'<tool_call>{{"tool":"calendar","args":{{'
            f'"action":"create","date":"{date}","title":"{title}"'
            f'}}}}</tool_call>'
        )

    else:
        prompts = [
            f"What do I have on {date}?",
            f"Show calendar {date}",
            f"Events on {date}",
            f"{date} schedule",
            f"mera calendar {date}",
            f"agenda para {date}"
        ]

        tool = (
            f'<tool_call>{{"tool":"calendar","args":{{'
            f'"action":"list","date":"{date}","title":""'
            f'}}}}</tool_call>'
        )

    return random.choice(prompts), tool


def convert_sample():
    val = round(random.uniform(1, 999), 2)
    from_u, to_u = random.sample(measure_units, 2)

    prompts = [
        f"Convert {val} {from_u} to {to_u}",
        f"{val} {from_u} in {to_u}",
        f"How many {to_u} is {val} {from_u}",
        f"{val} {from_u} ko {to_u} mein",
        f"cambiar {val} {from_u} a {to_u}"
    ]

    tool = (
        f'<tool_call>{{"tool":"convert","args":{{'
        f'"value":{val},"from_unit":"{from_u}","to_unit":"{to_u}"'
        f'}}}}</tool_call>'
    )

    return random.choice(prompts), tool


def currency_sample():
    amt = round(random.uniform(5, 5000), 2)
    c1, c2 = random.sample(currencies, 2)

    prompts = [
        f"Convert {amt} {c1} to {c2}",
        f"{amt} {c1} in {c2}",
        f"Exchange {amt} {c1} into {c2}",
        f"{amt} {c1} ka {c2}",
        f"cambiar {amt} {c1} a {c2}"
    ]

    tool = (
        f'<tool_call>{{"tool":"currency","args":{{'
        f'"amount":{amt},"from_ISO3":"{c1}","to_ISO3":"{c2}"'
        f'}}}}</tool_call>'
    )

    return random.choice(prompts), tool


def sql_sample():
    table = random.choice(sql_tables)
    limit = random.randint(3, 100)

    prompts = [
        f"Get all rows from {table}",
        f"SQL for top {limit} rows of {table}",
        f"select records from {table}",
        f"Need query for {table}",
        f"{table} table ka query"
    ]

    query_styles = [
        f"SELECT * FROM {table} LIMIT {limit};",
        f"SELECT id,name FROM {table} LIMIT {limit};",
        f"SELECT COUNT(*) FROM {table};",
        f"SELECT * FROM {table} ORDER BY id DESC LIMIT {limit};"
    ]

    query = random.choice(query_styles)

    tool = (
        f'<tool_call>{{"tool":"sql","args":{{'
        f'"query":"{escape_json(query)}"'
        f'}}}}</tool_call>'
    )

    return random.choice(prompts), tool

# ==========================================================
# GENERATORS
# ==========================================================
tool_generators = {
    "weather": weather_sample,
    "calendar": calendar_sample,
    "convert": convert_sample,
    "currency": currency_sample,
    "sql": sql_sample
}

# ==========================================================
# BUILD DATASET
# ==========================================================
dataset = []
seen = set()

# ---------- Balanced valid tool calls ----------
for tool_name, fn in tool_generators.items():
    count = 0
    while count < PER_TOOL:
        user, assistant = fn()

        key = user + assistant
        if key in seen:
            continue
        seen.add(key)

        dataset.append(write_jsonl_line(user, assistant))
        count += 1

# ---------- Refusals ----------
while len(dataset) < TOTAL_SAMPLES:
    user = random.choice(refusal_prompts)

    # add variations
    prefixes = ["", "Please ", "Hey, ", "Urgent: ", "Quickly ", ""]
    suffixes = ["", " now", " please", " asap", ""]
    user = random.choice(prefixes) + user + random.choice(suffixes)

    assistant = random.choice(refusal_responses)

    key = user + assistant
    if key in seen:
        continue

    seen.add(key)
    dataset.append(write_jsonl_line(user, assistant))

# ---------- Shuffle ----------
random.shuffle(dataset)

# ==========================================================
# SAVE
# ==========================================================
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for row in dataset:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# ==========================================================
# SUMMARY
# ==========================================================
print("Saved:", OUTPUT_FILE)
print("Total Samples:", len(dataset))
print("Valid Tool Calls:", VALID_SAMPLES)
print("Refusals:", REFUSAL_SAMPLES)
print("Balanced per tool:", PER_TOOL)

Saved: tool_router_700_balanced.jsonl
Total Samples: 2000
Valid Tool Calls: 1500
Refusals: 500
Balanced per tool: 300


In [13]:
!pip install -q transformers peft trl datasets bitsandbytes

# Training Script

In [14]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# 1. Load your dataset
dataset = load_dataset("json", data_files="/content/tool_router_700_balanced.jsonl", split="train")

# 2. Define the tiny model
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# 3. 4-Bit Quantization (This shrinks the model so it fits in the Colab GPU memory)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading Tokenizer and Model...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 4. Format the data to match Qwen's Chat format
def format_chat(example):
    example["text"] = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return example

dataset = dataset.map(format_chat)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 5. LoRA Config (We are training a lightweight "Adapter", not the whole model)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)

# 6. Training Setup (Using SFTConfig)
training_args = SFTConfig(
    output_dir="./pocket_agent_adapter",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=500,
    dataset_text_field="text",  # <--- Moved BACK to SFTConfig where the new update wants it
    max_length=512,             # <--- THE FIX: They renamed it from max_seq_length to max_length
)

# 7. Start the Engine
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=training_args,
    processing_class=tokenizer,
)

print("Starting Fine-Tuning...")
trainer.train()

# 8. Save the results!
trainer.save_model("my_trained_adapter")
print("Training Complete! Adapter saved to /my_trained_adapter")

Generating train split: 0 examples [00:00, ? examples/s]

Loading Tokenizer and Model...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting Fine-Tuning...


Step,Training Loss
10,3.573442
20,2.164479
30,1.461500
40,1.170635
50,0.954538
60,0.825617
70,0.636290
80,0.576950
90,0.521802
100,0.488976


Training Complete! Adapter saved to /my_trained_adapter


# Testing Trained Model

In [16]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Loading merged model for a quick sanity check...")
model_path = "./merged_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# The Test Prompt with a STRICT System Instruction
# The Test Prompt with the STRICT Blueprints
SYSTEM_PROMPT = """You are a strict tool-calling agent. Output ONLY valid JSON wrapped in <tool_call>...</tool_call> tags.
Strict Tool Schemas:
- weather: {"tool": "weather", "args": {"location": "string", "unit": "C|F"}}
- calendar: {"tool": "calendar", "args": {"action": "list|create", "date": "YYYY-MM-DD", "title": "string"}}
- convert: {"tool": "convert", "args": {"value": "number", "from_unit": "string", "to_unit": "string"}}
- currency: {"tool": "currency", "args": {"amount": "number", "from": "ISO3", "to": "ISO3"}}
- sql: {"tool": "sql", "args": {"query": "string"}}"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Convert 50 miles to kilometers."}
]

# Format it exactly how the model saw it during training
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

print("Generating response...")
# Generate the output
outputs = model.generate(**inputs, max_new_tokens=60, temperature=0.1)

# Decode only the newly generated text
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print("\n--- MODEL OUTPUT ---")
print(response)
#print("--------------------")

Loading merged model for a quick sanity check...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Generating response...

--- MODEL OUTPUT ---
<tool_call>{"tool":"convert","args":{"value":50,"from_unit":"Miles","to_unit":"Kilometers"}}</tool_call>
--------------------


# Merging Files

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_id = "Qwen/Qwen2.5-0.5B-Instruct"
adapter_path = "/content/my_trained_adapter"
output_dir = "./merged_model"

print("1. Loading Base Model in 16-bit (bf16)...")
# Notice we DO NOT use BitsAndBytes 4-bit here. We need the full weights to merge.
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16,
    device_map="cpu"
)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

print("2. Loading Adapter...")
model = PeftModel.from_pretrained(base_model, adapter_path)

print("3. Fusing Adapter and Base Model...")
model = model.merge_and_unload()

print(f"4. Saving the unified model to {output_dir}...")
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print("Merge Complete! Ready for Quantization.")

`torch_dtype` is deprecated! Use `dtype` instead!


1. Loading Base Model in 16-bit (bf16)...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

2. Loading Adapter...
3. Fusing Adapter and Base Model...
4. Saving the unified model to ./merged_model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merge Complete! Ready for Quantization.


#LLAMA.CPP conversion and quantization Code

In [17]:
# 1. Clean up old or failed GGUF files to ensure we are using the new training
!rm -f ./merged_model_f16.gguf ./pocket_agent_q4.gguf

# 2. Build the quantization tool if it's not already there (CMake version)
!cd llama.cpp && cmake -B build && cmake --build build --config Release -j 4 --target llama-quantize

# 3. CONVERSION: Convert your 'merged_model' folder into the large f16 GGUF
# This is the step that actually reads your new training data
!python llama.cpp/convert_hf_to_gguf.py ./merged_model --outfile ./merged_model_f16.gguf --outtype f16

# 4. QUANTIZATION: Smash that 1GB file down to ~300MB
!./llama.cpp/build/bin/llama-quantize ./merged_model_f16.gguf ./pocket_agent_q4.gguf Q4_K_M

# 5. VERIFICATION: Check the final file size
print("\n" + "="*30)
print("FINAL MODEL STATUS")
print("="*30)
!ls -lh pocket_agent_q4.gguf

CMAKE_BUILD_TYPE=Release
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- ggml version: 0.9.11
-- ggml commit:  45cac7ca7
-- OpenSSL found: 3.0.2
-- Generating embedded license file for target: llama-common
-- Configuring done (0.7s)
-- Generating done (0.5s)
-- Build files have been written to: /content/llama.cpp/build
[  0%] Built target cpp-httplib
[  2%] Built target llama-common-base
[  6%] Built target ggml-base
[ 13%] Built target ggml-cpu
[ 15%] Built target ggml
[ 18%] Building CXX object src/CMakeFiles/llama.dir/llama-model.cpp.o
[ 20%] Building CXX object src/CMakeFiles/llama.dir/models/deci.cpp.o
[ 20%] Building CXX object src/CMakeFiles/llama.dir/models/deepseek.cpp.o
[ 20%] Building CXX object src/CMakeFiles/llama.dir/models/deepseek2.cpp.o
[ 20

# Inference.py

In [24]:
import os
from llama_cpp import Llama

# 1. INITIALIZATION: Load the model
MODEL_PATH = "/content/pocket_agent_q4.gguf"
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model file not found at {MODEL_PATH}")

llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=1024,
    n_threads=4,
    verbose=False
)

# 2. FINAL SYSTEM PROMPT: Complete few-shot grounding for all 5 tools
SYSTEM_PROMPT = """You are a strict tool-calling agent. Output ONLY valid JSON wrapped in <tool_call>...</tool_call> tags.

Tool Schemas:
- weather: {"tool": "weather", "args": {"location": "string", "unit": "C|F"}}
- calendar: {"tool": "calendar", "args": {"action": "list|create", "date": "YYYY-MM-DD", "title": "string"}}
- convert: {"tool": "convert", "args": {"value": number, "from_unit": "string", "to_unit": "string"}}
- currency: {"tool": "currency", "args": {"amount": number, "from": "ISO3", "to": "ISO3"}}
- sql: {"tool": "sql", "args": {"query": "string"}}

Examples:
User: What is the weather in London in Celsius?
Assistant: <tool_call>{"tool": "weather", "args": {"location": "London", "unit": "C"}}</tool_call>

User: Schedule a meeting for 2026-06-01 called Strategy Session.
Assistant: <tool_call>{"tool": "calendar", "args": {"action": "create", "date": "2026-06-01", "title": "Strategy Session"}}</tool_call>

User: Convert 5 kilometers to miles.
Assistant: <tool_call>{"tool": "convert", "args": {"value": 5, "from_unit": "kilometers", "to_unit": "miles"}}</tool_call>

User: How much is 100 USD in EUR?
Assistant: <tool_call>{"tool": "currency", "args": {"amount": 100, "from": "USD", "to": "EUR"}}</tool_call>

User: Show me all employees in the sales department.
Assistant: <tool_call>{"tool": "sql", "args": {"query": "SELECT * FROM employees WHERE department = 'sales';"}}</tool_call>

User: Tell me a joke.
Assistant: I cannot engage in casual conversation.

User: Who is the president of the US?
Assistant: I cannot answer general knowledge or factual questions.

Rules:
1. NEVER use 'currency' for physical distances like miles or km.
2. ONLY use 'sql' if the user explicitly asks to query a database, table, or system records. DO NOT use 'sql' for trivia or general knowledge.
3. For 'convert', use full unit names as mentioned (e.g., "miles", "kilometers").
4. Output ONLY the tool call or a brief refusal. No conversational filler."""

# 3. CORE RUN FUNCTION
def run(prompt: str, history: list[dict]) -> str:
    # Build ChatML prompt
    full_prompt = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"

    for entry in history:
        full_prompt += f"<|im_start|>{entry['role']}\n{entry['content']}<|im_end|>\n"

    full_prompt += f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"

    # Deterministic generation
    output = llm(
        full_prompt,
        max_tokens=150,
        temperature=0.0,
        stop=["<|im_end|>", "<|endoftext|>"]
    )

    return output["choices"][0]["text"].strip()

# 4. BATCH TEST (For your final verification)
if __name__ == "__main__":
    test_prompts = [
        "How many kilometers are in 25 miles?",
        "What's the weather like in Lahore?",
        "Convert 500 PKR to USD",
        "Add an event for 2026-12-25 called Christmas Dinner",
        "Select all users from the orders table",
        "Who is the president of the US?"
    ]

    print("--- FINAL SYSTEM TEST ---")
    for tp in test_prompts:
        print(f"User: {tp}")
        print(f"Assistant: {run(tp, [])}\n")

llama_context: n_ctx_seq (1024) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


--- FINAL SYSTEM TEST ---
User: How many kilometers are in 25 miles?
Assistant: <tool_call>{"tool": "convert", "args": {"value": 25, "from_unit": "miles", "to_unit": "kilometers"}}</tool_call>

User: What's the weather like in Lahore?
Assistant: <tool_call>{"tool": "weather", "args": {"location": "Lahore", "unit": "F"}}</tool_call>

User: Convert 500 PKR to USD
Assistant: <tool_call>{"tool": "convert", "args": {"value": 500, "from_unit": "PKR", "to_unit": "USD"}}</tool_call>

User: Add an event for 2026-12-25 called Christmas Dinner
Assistant: <tool_call>{"tool": "calendar", "args": {"action": "create", "date": "2026-12-25", "title": "Christmas Dinner"}}</tool_call>

User: Select all users from the orders table
Assistant: <tool_call>{"tool": "sql", "args": {"query": "SELECT * FROM users ORDER BY id DESC LIMIT 100;"}</tool_call>

User: Who is the president of the US?
Assistant: <tool_call>{"tool": "currency", "args": {"amount": 1, "from": "ISO3", "to": "ISO3"}}</tool_call>

